In [2]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 6.9 MB/s eta 0:00:00


In [3]:
import cv2
import pandas as pd
import os

from ultralytics import YOLO
from google.colab import files
from IPython.display import display, Video

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [4]:
model = YOLO("yolov8n.pt")

In [5]:
uploaded = files.upload()

Saving 133319-756469375_medium.mp4 to 133319-756469375_medium.mp4


In [6]:
video_path = list(uploaded.keys())[0]

print("Uploaded video:", video_path)

Uploaded video: 133319-756469375_medium.mp4


In [7]:
cap = cv2.VideoCapture(video_path)#open video

fps = cap.get(cv2.CAP_PROP_FPS)#getting frame per sec from video
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))#getting width of frame
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))#getting height of frame
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))#count total number of frames in the video

print("FPS:", fps)
print("Resolution:", width, "x", height)
print("Total frames:", total_frames)

FPS: 29.97002997002997
Resolution: 2560 x 1440
Total frames: 647


In [8]:
output_video = "detected_output.mp4" #filename of the processed output video

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    output_video,
    fourcc,
    fps,
    (width, height)
)

detections_data = []#creates an empty list to store detected objects
frame_number = 0#frame counter.
while cap.isOpened():#Keep processing while the input video is open.

    success, frame = cap.read()#checks whether OpenCV successfully read the fram

    if not success:
        break

    frame_number += 1#increase counting
    results = model(frame, verbose=False)#passing frames to model for training

    result = results[0]# stored in result
    annotated_frame = result.plot()

    for box in result.boxes:#loop runs once for each detected object

        class_id = int(box.cls[0])#represents objects using class number
        confidence = float(box.conf[0])#gets YOLO's confidence score

        object_name = model.names[class_id]#converts the numerical class ID into a readable object name

        x1, y1, x2, y2 = box.xyxy[0].tolist()#gets the coordinates of the bounding box.

        timestamp = frame_number / fps#calculates at what time in the video the object was detected

        detections_data.append({  #updating list with data
            "frame_number": frame_number,
            "timestamp_seconds": round(timestamp, 2),
            "object_name": object_name,
            "confidence": round(confidence, 4),
            "x1": round(x1, 2),
            "y1": round(y1, 2),
            "x2": round(x2, 2),
            "y2": round(y2, 2)
        })
    writer.write(annotated_frame)#writes the current processed frame into your output video

    if frame_number % 100 == 0:
        print(
            f"Processed {frame_number}/{total_frames} frames"
        )


cap.release()#closes and finalizes your output video
writer.release()#output video may be incomplete

print("Video processing completed.")

Processed 100/647 frames
Processed 200/647 frames
Processed 300/647 frames
Processed 400/647 frames
Processed 500/647 frames
Processed 600/647 frames
Video processing completed.


In [9]:
df = pd.DataFrame(detections_data)

csv_file = "detected_objects.csv"

df.to_csv(csv_file, index=False)

print("CSV created successfully.")

display(df.head(20))

CSV created successfully.


,frame_number,timestamp_seconds,object_name,confidence,x1,y1,x2,y2
0,1,0.03,cup,0.9688,552.76,307.64,1498.87,1242.95
1,1,0.03,remote,0.5841,1472.02,637.35,2345.02,1207.32
2,2,0.07,cup,0.9686,552.70,307.68,1498.85,1242.95
3,2,0.07,remote,0.6141,1460.27,633.27,2339.12,1206.39
4,3,0.10,cup,0.9677,552.92,307.63,1498.67,1242.76
5,3,0.10,remote,0.5960,1457.88,627.48,2339.89,1205.73
6,4,0.13,cup,0.9678,552.85,307.86,1498.77,1243.02
7,4,0.13,remote,0.5345,1462.19,630.15,2339.68,1205.76
8,5,0.17,cup,0.9688,552.24,307.97,1499.16,1243.00
9,5,0.17,remote,0.5187,1465.48,636.09,2340.11,1206.26
